In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

CUDA available: True
Device name: Tesla T4


In [ ]:
import os
from huggingface_hub import login

# Paste your token here
HF_TOKEN = ""
login(token=HF_TOKEN)

# Also set environment variable for datasets and transformers libraries
os.environ["HF_TOKEN"] = HF_TOKEN

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("fancyzhx/ag_news", trust_remote_code=True)
# If the above or "ag_news" has any issue, "fancyzhx/ag_news" is the exact canonical mirror on HF Hub

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

print("Tokenizing train and test sets...")
tokenized_datasets = dataset.map(preprocess_function, batched=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'fancyzhx/ag_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'fancyzhx/ag_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Tokenizing train and test sets...


In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {
        "accuracy": acc,
        "f1": f1
    }

In [11]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./distilbert-agnews-results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                       # Fast GPU training on T4
    logging_steps=200
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.176213,0.170709,0.941184,0.941166


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.176213,0.170709,0.941184,0.941166
2,0.132286,0.167997,0.946053,0.946125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7500, training_loss=0.17889470291137696, metrics={'train_runtime': 638.7521, 'train_samples_per_second': 375.733, 'train_steps_per_second': 11.742, 'total_flos': 6509878851667968.0, 'train_loss': 0.17889470291137696, 'epoch': 2.0})

In [12]:
results = trainer.evaluate()
print("\n" + "="*40)
print("DISTILBERT EVALUATION RESULTS:")
print("="*40)
for k, v in results.items():
    print(f"{k}: {v}")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.132286,0.167997,2,0.946053,0.946125



DISTILBERT EVALUATION RESULTS:
eval_loss: 0.16799671947956085
eval_accuracy: 0.9460526315789474
eval_f1: 0.9461251799362785
